# PMF full-period factor contribution time analysis

This notebook merges the selected full-period PMF source contributions to timestamps, adds Astral day/night, plots factor contribution time trends, identifies recurrent high-contribution PMF events, and optionally compares PMF factor spikes with manual event selection.

Working assumption: this notebook uses the **full-period 6-factor / 20-base-run / 5% extra modeling uncertainty** output. The lowest-Q converged base run from your diagnostics was Run 3.


In [1]:
# Optional, only if needed:
# !pip install astral

from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astral import Observer
from astral.sun import sun

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

# -----------------------------
# User paths
# -----------------------------
ROOT = Path(r"D:\Documents\PhD-Research\Xact python code")
PMF_DIR = ROOT / "pmf_outputs" / "full_6f_20runs"
SAMPLE_KEY_PATH = ROOT / "pmf_ready" / "Xact_PMF_full_excluding_downtime_outlier_sample_key.csv"

CONTRIB_PATH = PMF_DIR / "full_contributions.txt"
PROFILE_PATH = PMF_DIR / "full_profiles.txt"
DIAGNOSTICS_PATH = PMF_DIR / "full_diagnostics.txt"
RUN_COMPARISON_PATH = PMF_DIR / "full_run_comparison.txt"

OUT_DIR = PMF_DIR / "_time_analysis"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ASCENT Pittsburgh / Lawrenceville site
LAT = 40.46542
LON = -79.960757
TZ = "America/New_York"

# Factor labels for the 20-run 6-factor lowest-Q Run 3 solution.
# Change only if you intentionally switch to a different PMF base run.
FACTOR_LABELS = {
    "Factor1": "Fe-Mn-rich industrial metal",
    "Factor2": "K-As-rich mixed combustion/event",
    "Factor3": "Ca-Fe-Ti dust/urban background",
    "Factor4": "Zn-rich industrial plume",
    "Factor5": "S-rich regional/background",
    "Factor6": "Cu-Ba/Sr-rich event",
}

print("PMF_DIR:", PMF_DIR)
print("Sample key:", SAMPLE_KEY_PATH)
print("Output folder:", OUT_DIR)


PMF_DIR: D:\Documents\PhD-Research\Xact python code\pmf_outputs\full_6f_20runs
Sample key: D:\Documents\PhD-Research\Xact python code\pmf_ready\Xact_PMF_full_excluding_downtime_outlier_sample_key.csv
Output folder: D:\Documents\PhD-Research\Xact python code\pmf_outputs\full_6f_20runs\_time_analysis


## 1. Helper functions


In [3]:
def sample_num_from_value(x):
    """Convert SampleID formats like ID1, ID001, 1, 1.0 into integer 1."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    m = re.search(r"(\d+)", s)
    if not m:
        return np.nan
    return int(m.group(1))


def get_factor_cols(df):
    return sorted([c for c in df.columns if re.match(r"^Factor\d+$", c)], key=lambda c: int(c.replace("Factor", "")))


def read_pmf_contributions_txt(path):
    """Parse EPA PMF contribution text output into a tidy dataframe."""
    path = Path(path)
    lines = path.read_text(errors="ignore").splitlines()

    header_idx = None
    factor_names = None
    for i, line in enumerate(lines):
        if "Factor 1" in line and "Factor 2" in line:
            header_idx = i
            factor_names = [f"Factor{n}" for n in re.findall(r"Factor\s+(\d+)", line)]
            break

    if header_idx is None or not factor_names:
        raise ValueError(f"Could not find factor header in {path}")

    rows = []
    for line in lines[header_idx + 1:]:
        if not line.strip():
            continue
        parts = re.split(r"\s+", line.strip())
        if len(parts) < 2 + len(factor_names):
            continue
        if not re.match(r"^\d+$", parts[0]):
            continue
        if not re.match(r"^(ID)?\d+$", parts[1], flags=re.IGNORECASE):
            continue
        try:
            values = [float(v) for v in parts[2:2 + len(factor_names)]]
        except ValueError:
            continue
        rows.append([int(parts[0]), parts[1], sample_num_from_value(parts[1]), *values])

    if not rows:
        raise ValueError(f"No contribution rows parsed from {path}")

    df = pd.DataFrame(rows, columns=["base_run", "SampleID_pmf", "sample_num", *factor_names])
    return df[["base_run", "SampleID_pmf", "sample_num", *get_factor_cols(df)]]


def infer_sample_key_columns(sample_key):
    """Infer sample ID and timestamp columns from the sample key dataframe."""
    cols = list(sample_key.columns)
    lower = {c: str(c).strip().lower() for c in cols}

    sample_candidates = [c for c in cols if "sample" in lower[c] and "id" in lower[c]]
    if not sample_candidates:
        sample_candidates = [c for c in cols if "id" in lower[c]]
    if not sample_candidates:
        raise ValueError("Could not infer SampleID column. Set sample_col manually.")
    sample_col = sample_candidates[0]

    utc_candidates = [
        c for c in cols
        if c != sample_col
        and "utc" in lower[c]
        and any(tok in lower[c] for tok in ["time", "date", "datetime", "timestamp"])
    ]
    if utc_candidates:
        best_col, best_count = None, -1
        for c in utc_candidates:
            parsed = pd.to_datetime(sample_key[c], errors="coerce", utc=True)
            count = parsed.notna().sum()
            if count > best_count:
                best_col, best_count = c, count
        if best_col is not None and best_count > 0:
            return sample_col, best_col, "utc"

    time_candidates = [c for c in cols if c != sample_col and any(tok in lower[c] for tok in ["time", "date", "datetime", "timestamp"])]
    if not time_candidates:
        time_candidates = [c for c in cols if c != sample_col]

    best_col, best_count = None, -1
    for c in time_candidates:
        parsed = pd.to_datetime(sample_key[c], errors="coerce")
        count = parsed.notna().sum()
        if count > best_count:
            best_col, best_count = c, count

    if best_col is None or best_count == 0:
        raise ValueError("Could not infer timestamp column. Set time_col manually.")
    return sample_col, best_col, "local"


def localize_or_convert_time(series, tz=TZ, source_tz="local"):
    """Return timezone-aware local timestamps; UTC columns avoid DST ambiguous-hour loss."""
    if source_tz == "utc":
        return pd.to_datetime(series, errors="coerce", utc=True).dt.tz_convert(tz)

    ts = pd.to_datetime(series, errors="coerce")
    if getattr(ts.dt, "tz", None) is None:
        return ts.dt.tz_localize(tz, nonexistent="shift_forward", ambiguous="NaT")
    return ts.dt.tz_convert(tz)


def read_sample_key(path):
    key = pd.read_csv(path)
    key.columns = [str(c).strip() for c in key.columns]
    sample_col, time_col, time_source = infer_sample_key_columns(key)
    print(f"Inferred sample key columns: sample_col={sample_col!r}, time_col={time_col!r}, time_source={time_source!r}")

    # EPA PMF contribution outputs label rows as ID1..IDN by PMF input order, even when
    # the input SampleID column preserves original master-row IDs with gaps.
    key["pmf_row"] = np.arange(1, len(key) + 1)
    key["sample_num"] = key[sample_col].apply(sample_num_from_value)
    key["timestamp_local"] = localize_or_convert_time(key[time_col], tz=TZ, source_tz=time_source)
    key = key.dropna(subset=["pmf_row", "timestamp_local"]).copy()
    key["pmf_row"] = key["pmf_row"].astype(int)
    key["sample_num"] = key["sample_num"].astype("Int64")
    return key, sample_col, time_col


def add_astral_daynight(df, time_col="timestamp_local", lat=LAT, lon=LON, tz=TZ):
    out = df.copy()
    observer = Observer(latitude=lat, longitude=lon)
    out["date_local"] = out[time_col].dt.date

    sun_rows = []
    for d in sorted(out["date_local"].dropna().unique()):
        s = sun(observer, date=d, tzinfo=tz)
        sun_rows.append({"date_local": d, "sunrise": s["sunrise"], "sunset": s["sunset"]})
    sun_df = pd.DataFrame(sun_rows)

    out = out.merge(sun_df, on="date_local", how="left")
    out["is_day"] = (out[time_col] >= out["sunrise"]) & (out[time_col] < out["sunset"])
    out["daynight"] = np.where(out["is_day"], "day", "night")
    return out


def add_temporal_fields(df, time_col="timestamp_local"):
    out = df.copy()
    t = out[time_col]
    out["date"] = t.dt.date
    out["year"] = t.dt.year
    out["month"] = t.dt.month
    out["month_name"] = t.dt.strftime("%Y-%m")
    out["hour"] = t.dt.hour
    out["dayofweek"] = t.dt.dayofweek
    out["weekday_name"] = t.dt.day_name()
    out["is_weekend"] = out["dayofweek"].isin([5, 6])
    out["weekday_weekend"] = np.where(out["is_weekend"], "weekend", "weekday")
    out["season"] = out["month"].map({12:"winter", 1:"winter", 2:"winter", 3:"spring", 4:"spring", 5:"spring", 6:"summer", 7:"summer", 8:"summer", 9:"fall", 10:"fall", 11:"fall"})
    return out


def add_known_windows(df, time_col="timestamp_local"):
    out = df.copy()
    known_windows = [
        {"window": "wildfire_2023_extended", "start": "2023-06-27 00:00", "end": "2023-07-07 23:59"},
        {"window": "fireworks_2023", "start": "2023-06-30 00:00", "end": "2023-07-06 23:59"},
        {"window": "fireworks_2024", "start": "2024-06-29 00:00", "end": "2024-07-01 23:59"},
    ]
    out["known_event_window"] = "none"
    for w in known_windows:
        start = pd.Timestamp(w["start"], tz=TZ)
        end = pd.Timestamp(w["end"], tz=TZ)
        mask = (out[time_col] >= start) & (out[time_col] <= end)
        out.loc[mask & out["known_event_window"].eq("none"), "known_event_window"] = w["window"]
        overlap = mask & out["known_event_window"].ne("none") & out["known_event_window"].ne(w["window"])
        out.loc[overlap, "known_event_window"] = out.loc[overlap, "known_event_window"] + ";" + w["window"]
    out["in_known_event_window"] = out["known_event_window"].ne("none")
    return out


## 2. Read and merge PMF contributions to timestamp


In [4]:
contrib = read_pmf_contributions_txt(CONTRIB_PATH)
sample_key, sample_col, time_col = read_sample_key(SAMPLE_KEY_PATH)
factor_cols = get_factor_cols(contrib)

print("Contributions shape:", contrib.shape)
print("Sample key shape:", sample_key.shape)
print("Factors:", factor_cols)

display(contrib.head())
display(sample_key[[sample_col, "pmf_row", time_col, "timestamp_local"]].head())

key_cols = ["pmf_row", sample_col, "timestamp_local"]
for optional_col in ["Master_Row", "Date_Local", "Date_UTC"]:
    if optional_col in sample_key.columns and optional_col not in key_cols:
        key_cols.append(optional_col)

key_for_merge = sample_key[key_cols].copy()
if sample_col != "SampleID_input":
    key_for_merge = key_for_merge.rename(columns={sample_col: "SampleID_input"})

merged = contrib.merge(
    key_for_merge,
    left_on="sample_num",
    right_on="pmf_row",
    how="left",
    validate="one_to_one"
)

missing_time = merged["timestamp_local"].isna().sum()
print(f"Rows after merge: {len(merged):,}")
print(f"Missing timestamps after merge: {missing_time:,}")
print("Merge key: PMF contribution ID number -> sample-key row position (pmf_row)")
if missing_time > 0:
    display(merged.loc[merged["timestamp_local"].isna()].head(20))
    warnings.warn("Some contribution rows did not match the sample-key row position. Check PMF input/output row counts.")

merged = merged.dropna(subset=["timestamp_local"]).sort_values("timestamp_local").reset_index(drop=True)
merged = add_astral_daynight(merged, time_col="timestamp_local")
merged = add_temporal_fields(merged, time_col="timestamp_local")
merged = add_known_windows(merged, time_col="timestamp_local")

merged_path = OUT_DIR / "pmf_contributions_merged_timestamp_daynight.csv"
merged.to_csv(merged_path, index=False)

print("Saved:", merged_path)
display(merged.head())
display(merged.tail())


Inferred sample key columns: sample_col='SampleID', time_col='Date_UTC', time_source='utc'
Contributions shape: (18094, 9)
Sample key shape: (18094, 8)
Factors: ['Factor1', 'Factor2', 'Factor3', 'Factor4', 'Factor5', 'Factor6']


,base_run,SampleID_pmf,sample_num,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6
0,3,ID1,1,0.44549,0.341720,1.83090,1.763800,0.057013,0.397900
1,3,ID2,2,1.38840,-0.005368,0.76260,0.981190,0.057616,0.071118
2,3,ID3,3,0.11614,0.080668,0.23935,0.427200,0.039859,1.292200
3,3,ID4,4,0.25282,0.018697,0.20632,0.068922,0.027314,0.535140
4,3,ID5,5,0.07238,0.214930,0.49923,0.189710,-0.001660,1.998900


,SampleID,pmf_row,Date_UTC,timestamp_local
0,1,1,2023-05-02 16:00:00,2023-05-02 12:00:00-04:00
1,2,2,2023-05-02 17:00:00,2023-05-02 13:00:00-04:00
2,3,3,2023-05-02 18:00:00,2023-05-02 14:00:00-04:00
3,4,4,2023-05-02 19:00:00,2023-05-02 15:00:00-04:00
4,5,5,2023-05-02 20:00:00,2023-05-02 16:00:00-04:00


Rows after merge: 18,094
Missing timestamps after merge: 0
Merge key: PMF contribution ID number -> sample-key row position (pmf_row)
Saved: D:\Documents\PhD-Research\Xact python code\pmf_outputs\full_6f_20runs\_time_analysis\pmf_contributions_merged_timestamp_daynight.csv


,base_run,SampleID_pmf,sample_num,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,pmf_row,SampleID_input,timestamp_local,Master_Row,Date_Local,Date_UTC,date_local,sunrise,sunset,is_day,daynight,date,year,month,month_name,hour,dayofweek,weekday_name,is_weekend,weekday_weekend,season,known_event_window,in_known_event_window
0,3,ID1,1,0.44549,0.341720,1.83090,1.763800,0.057013,0.397900,1,1,2023-05-02 12:00:00-04:00,1,2023-05-02 12:00:00,2023-05-02 16:00:00,2023-05-02,2023-05-02 06:18:12.444440-04:00,2023-05-02 20:16:07.475881-04:00,True,day,2023-05-02,2023,5,2023-05,12,1,Tuesday,False,weekday,spring,none,False
1,3,ID2,2,1.38840,-0.005368,0.76260,0.981190,0.057616,0.071118,2,2,2023-05-02 13:00:00-04:00,2,2023-05-02 13:00:00,2023-05-02 17:00:00,2023-05-02,2023-05-02 06:18:12.444440-04:00,2023-05-02 20:16:07.475881-04:00,True,day,2023-05-02,2023,5,2023-05,13,1,Tuesday,False,weekday,spring,none,False
2,3,ID3,3,0.11614,0.080668,0.23935,0.427200,0.039859,1.292200,3,3,2023-05-02 14:00:00-04:00,3,2023-05-02 14:00:00,2023-05-02 18:00:00,2023-05-02,2023-05-02 06:18:12.444440-04:00,2023-05-02 20:16:07.475881-04:00,True,day,2023-05-02,2023,5,2023-05,14,1,Tuesday,False,weekday,spring,none,False
3,3,ID4,4,0.25282,0.018697,0.20632,0.068922,0.027314,0.535140,4,4,2023-05-02 15:00:00-04:00,4,2023-05-02 15:00:00,2023-05-02 19:00:00,2023-05-02,2023-05-02 06:18:12.444440-04:00,2023-05-02 20:16:07.475881-04:00,True,day,2023-05-02,2023,5,2023-05,15,1,Tuesday,False,weekday,spring,none,False
4,3,ID5,5,0.07238,0.214930,0.49923,0.189710,-0.001660,1.998900,5,5,2023-05-02 16:00:00-04:00,5,2023-05-02 16:00:00,2023-05-02 20:00:00,2023-05-02,2023-05-02 06:18:12.444440-04:00,2023-05-02 20:16:07.475881-04:00,True,day,2023-05-02,2023,5,2023-05,16,1,Tuesday,False,weekday,spring,none,False


,base_run,SampleID_pmf,sample_num,Factor1,Factor2,Factor3,Factor4,Factor5,Factor6,pmf_row,SampleID_input,timestamp_local,Master_Row,Date_Local,Date_UTC,date_local,sunrise,sunset,is_day,daynight,date,year,month,month_name,hour,dayofweek,weekday_name,is_weekend,weekday_weekend,season,known_event_window,in_known_event_window
18089,3,ID18090,18090,4.0011,2.4586,1.7475,0.69995,0.58320,2.2597,18090,18971,2025-10-07 01:00:00-04:00,18971,2025-10-07 01:00:00,2025-10-07 05:00:00,2025-10-07,2025-10-07 07:22:56.945597-04:00,2025-10-07 18:51:29.514695-04:00,False,night,2025-10-07,2025,10,2025-10,1,1,Tuesday,False,weekday,fall,none,False
18090,3,ID18091,18091,4.5973,1.6029,2.1435,1.65070,0.70582,0.2520,18091,18972,2025-10-07 02:00:00-04:00,18972,2025-10-07 02:00:00,2025-10-07 06:00:00,2025-10-07,2025-10-07 07:22:56.945597-04:00,2025-10-07 18:51:29.514695-04:00,False,night,2025-10-07,2025,10,2025-10,2,1,Tuesday,False,weekday,fall,none,False
18091,3,ID18092,18092,6.5435,1.6694,2.4891,2.75890,0.60045,2.6436,18092,18973,2025-10-07 03:00:00-04:00,18973,2025-10-07 03:00:00,2025-10-07 07:00:00,2025-10-07,2025-10-07 07:22:56.945597-04:00,2025-10-07 18:51:29.514695-04:00,False,night,2025-10-07,2025,10,2025-10,3,1,Tuesday,False,weekday,fall,none,False
18092,3,ID18093,18093,8.4129,1.8371,3.6224,4.88870,0.71253,2.5106,18093,18974,2025-10-07 04:00:00-04:00,18974,2025-10-07 04:00:00,2025-10-07 08:00:00,2025-10-07,2025-10-07 07:22:56.945597-04:00,2025-10-07 18:51:29.514695-04:00,False,night,2025-10-07,2025,10,2025-10,4,1,Tuesday,False,weekday,fall,none,False
18093,3,ID18094,18094,10.3420,2.5378,3.3966,10.67300,1.07720,1.0024,18094,18975,2025-10-07 05:00:00-04:00,18975,2025-10-07 05:00:00,2025-10-07 09:00:00,2025-10-07,2025-10-07 07:22:56.945597-04:00,2025-10-07 18:51:29.514695-04:00,False,night,2025-10-07,2025,10,2025-10,5,1,Tuesday,False,weekday,fall,none,False


## 3. Quick QA


In [ ]:
print("Time range:", merged["timestamp_local"].min(), "to", merged["timestamp_local"].max())
print("\nBase run IDs in contribution file:")
display(merged["base_run"].value_counts().sort_index().to_frame("n"))

print("\nDay/night counts:")
display(merged["daynight"].value_counts(dropna=False).to_frame("n_hours"))

print("\nKnown event-window hours:")
display(merged["known_event_window"].value_counts(dropna=False).to_frame("n_hours"))

print("\nFactor contribution summary:")
display(merged[factor_cols].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).T)


## 4. Factor contribution time trends


In [ ]:
# Daily mean contribution for long-term trend
# The raw hourly plot is below for one selected factor.
daily = merged.set_index("timestamp_local")[factor_cols].resample("D").mean().reset_index()

for f in factor_cols:
    label = FACTOR_LABELS.get(f, f)
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(daily["timestamp_local"], daily[f], linewidth=1)
    ax.set_title(f"{f}: {label} — daily mean PMF contribution")
    ax.set_xlabel("Date")
    ax.set_ylabel("Daily mean contribution")
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{f}_daily_mean_timeseries.png", dpi=200)
    plt.show()

print("Saved plots to:", FIG_DIR)


In [ ]:
# Raw hourly plot for one factor. Change SELECT_FACTOR to inspect each factor.
SELECT_FACTOR = "Factor4"  # Factor1 Fe-Mn, Factor4 Zn, Factor6 Cu-Ba/Sr, etc.

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(merged["timestamp_local"], merged[SELECT_FACTOR], linewidth=0.6)
ax.set_title(f"{SELECT_FACTOR}: {FACTOR_LABELS.get(SELECT_FACTOR, SELECT_FACTOR)} — hourly contribution")
ax.set_xlabel("Date")
ax.set_ylabel("Hourly contribution")
fig.autofmt_xdate()
fig.tight_layout()
plt.show()


## 5. Define high-contribution PMF hours


In [ ]:
def add_factor_quantile_flags(df, factors, quantiles=(0.90, 0.95, 0.99)):
    out = df.copy()
    rows = []
    for f in factors:
        vals = out[f].replace([-np.inf, np.inf], np.nan)
        for q in quantiles:
            thr = vals.quantile(q)
            suffix = f"top{int(round((1-q)*100)):02d}"  # top05 for 95th, top01 for 99th
            col = f"{f}_{suffix}"
            out[col] = out[f] >= thr
            rows.append({
                "factor": f,
                "factor_label": FACTOR_LABELS.get(f, f),
                "quantile": q,
                "threshold": thr,
                "flag_col": col,
                "n_hours": int(out[col].sum()),
            })
    return out, pd.DataFrame(rows)

merged_flags, factor_thresholds = add_factor_quantile_flags(merged, factor_cols, quantiles=(0.90, 0.95, 0.99))
factor_thresholds_path = OUT_DIR / "factor_high_contribution_thresholds.csv"
factor_thresholds.to_csv(factor_thresholds_path, index=False)

display(factor_thresholds)
print("Saved:", factor_thresholds_path)


In [ ]:
def high_hour_summary(df, factors, top_col_suffix="top05"):
    rows = []
    for f in factors:
        flag_col = f"{f}_{top_col_suffix}"
        sub = df[df[flag_col]].copy()
        rows.append({
            "factor": f,
            "factor_label": FACTOR_LABELS.get(f, f),
            "top_definition": top_col_suffix,
            "n_high_hours": len(sub),
            "start": sub["timestamp_local"].min(),
            "end": sub["timestamp_local"].max(),
            "median_contribution": sub[f].median(),
            "max_contribution": sub[f].max(),
            "pct_day": 100 * sub["is_day"].mean() if len(sub) else np.nan,
            "pct_night": 100 * (~sub["is_day"]).mean() if len(sub) else np.nan,
            "pct_weekend": 100 * sub["is_weekend"].mean() if len(sub) else np.nan,
            "pct_in_known_event_window": 100 * sub["in_known_event_window"].mean() if len(sub) else np.nan,
        })
    return pd.DataFrame(rows)

summary_top05 = high_hour_summary(merged_flags, factor_cols, "top05")
summary_top01 = high_hour_summary(merged_flags, factor_cols, "top01")

summary_top05.to_csv(OUT_DIR / "factor_high_hour_summary_top05.csv", index=False)
summary_top01.to_csv(OUT_DIR / "factor_high_hour_summary_top01.csv", index=False)

display(summary_top05)
display(summary_top01)


## 6. Group consecutive high-contribution hours into PMF factor events


In [ ]:
def build_factor_events(df, factor, top_col_suffix="top05", max_gap_hours=1.5, min_event_hours=1):
    flag_col = f"{factor}_{top_col_suffix}"
    sub = df[df[flag_col]].copy().sort_values("timestamp_local")
    if sub.empty:
        return pd.DataFrame()

    gap_h = sub["timestamp_local"].diff().dt.total_seconds() / 3600
    sub["new_event"] = gap_h.isna() | (gap_h > max_gap_hours)
    sub["event_index"] = sub["new_event"].cumsum()

    rows = []
    for idx, g in sub.groupby("event_index"):
        if len(g) < min_event_hours:
            continue
        rows.append({
            "factor": factor,
            "factor_label": FACTOR_LABELS.get(factor, factor),
            "top_definition": top_col_suffix,
            "event_id": f"{factor}_{top_col_suffix}_E{int(idx):04d}",
            "start": g["timestamp_local"].min(),
            "end": g["timestamp_local"].max(),
            "n_hours": len(g),
            "duration_hours_inclusive": (g["timestamp_local"].max() - g["timestamp_local"].min()).total_seconds()/3600 + 1,
            "mean_contribution": g[factor].mean(),
            "max_contribution": g[factor].max(),
            "pct_day": 100 * g["is_day"].mean(),
            "pct_night": 100 * (~g["is_day"]).mean(),
            "pct_weekend": 100 * g["is_weekend"].mean(),
            "season_mode": g["season"].mode().iat[0] if not g["season"].mode().empty else np.nan,
            "month_start": g["month_name"].iloc[0],
            "known_event_window_mode": g["known_event_window"].mode().iat[0] if not g["known_event_window"].mode().empty else "none",
            "in_known_event_window_any": bool(g["in_known_event_window"].any()),
        })
    return pd.DataFrame(rows)

events_top05 = pd.concat([build_factor_events(merged_flags, f, "top05", max_gap_hours=1.5, min_event_hours=1) for f in factor_cols], ignore_index=True)
events_top01 = pd.concat([build_factor_events(merged_flags, f, "top01", max_gap_hours=1.5, min_event_hours=1) for f in factor_cols], ignore_index=True)

events_top05.to_csv(OUT_DIR / "factor_events_top05.csv", index=False)
events_top01.to_csv(OUT_DIR / "factor_events_top01.csv", index=False)

print("Top 5% events:", events_top05.shape)
display(events_top05.sort_values(["factor", "start"]).head(30))
print("Top 1% events:", events_top01.shape)
display(events_top01.sort_values(["factor", "start"]).head(30))


## 7. Day/night, hour-of-day, month, and season behavior


In [ ]:
long = merged_flags.melt(
    id_vars=["sample_num", "SampleID_pmf", "timestamp_local", "date", "year", "month", "month_name", "season", "hour", "daynight", "is_day", "is_weekend", "weekday_weekend", "known_event_window", "in_known_event_window"],
    value_vars=factor_cols,
    var_name="factor",
    value_name="contribution",
)
long["factor_label"] = long["factor"].map(FACTOR_LABELS).fillna(long["factor"])

for suffix in ["top05", "top01"]:
    flag_long = []
    for f in factor_cols:
        tmp = merged_flags[["sample_num", f"{f}_{suffix}"]].copy()
        tmp["factor"] = f
        tmp = tmp.rename(columns={f"{f}_{suffix}": suffix})
        flag_long.append(tmp)
    long = long.merge(pd.concat(flag_long, ignore_index=True), on=["sample_num", "factor"], how="left")

daynight_summary = long.groupby(["factor", "factor_label", "daynight"], dropna=False).agg(
    n_hours=("contribution", "size"),
    mean_contribution=("contribution", "mean"),
    median_contribution=("contribution", "median"),
    p95_contribution=("contribution", lambda x: x.quantile(0.95)),
    n_top05=("top05", "sum"),
    n_top01=("top01", "sum"),
).reset_index()

daynight_summary.to_csv(OUT_DIR / "factor_daynight_summary.csv", index=False)
display(daynight_summary)


In [ ]:
hour_summary = long.groupby(["factor", "factor_label", "hour"], dropna=False).agg(
    mean_contribution=("contribution", "mean"),
    median_contribution=("contribution", "median"),
    p95_contribution=("contribution", lambda x: x.quantile(0.95)),
    n_top05=("top05", "sum"),
    n_hours=("contribution", "size"),
).reset_index()
hour_summary.to_csv(OUT_DIR / "factor_hour_of_day_summary.csv", index=False)

for f in factor_cols:
    sub = hour_summary[hour_summary["factor"] == f]
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(sub["hour"], sub["p95_contribution"], marker="o")
    ax.set_title(f"{f}: {FACTOR_LABELS.get(f, f)} — p95 contribution by hour")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel("p95 contribution")
    ax.set_xticks(range(0, 24, 2))
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{f}_hour_of_day_p95.png", dpi=200)
    plt.show()


In [ ]:
month_summary = long.groupby(["factor", "factor_label", "month_name"], dropna=False).agg(
    n_hours=("contribution", "size"),
    mean_contribution=("contribution", "mean"),
    p95_contribution=("contribution", lambda x: x.quantile(0.95)),
    n_top05=("top05", "sum"),
    n_top01=("top01", "sum"),
).reset_index()

season_summary = long.groupby(["factor", "factor_label", "season"], dropna=False).agg(
    n_hours=("contribution", "size"),
    mean_contribution=("contribution", "mean"),
    p95_contribution=("contribution", lambda x: x.quantile(0.95)),
    n_top05=("top05", "sum"),
    n_top01=("top01", "sum"),
).reset_index()

month_summary.to_csv(OUT_DIR / "factor_month_summary.csv", index=False)
season_summary.to_csv(OUT_DIR / "factor_season_summary.csv", index=False)

display(season_summary)


## 8. Known wildfire/fireworks window overlap


In [ ]:
known_overlap_rows = []
for f in factor_cols:
    for suffix in ["top05", "top01"]:
        flag_col = f"{f}_{suffix}"
        sub = merged_flags[merged_flags[flag_col]].copy()
        for w, g in sub.groupby("known_event_window", dropna=False):
            known_overlap_rows.append({
                "factor": f,
                "factor_label": FACTOR_LABELS.get(f, f),
                "top_definition": suffix,
                "known_event_window": w,
                "n_high_hours": len(g),
                "pct_of_factor_high_hours": 100 * len(g) / len(sub) if len(sub) else np.nan,
            })

known_overlap = pd.DataFrame(known_overlap_rows)
known_overlap.to_csv(OUT_DIR / "factor_known_window_overlap.csv", index=False)
display(known_overlap.sort_values(["factor", "top_definition", "n_high_hours"], ascending=[True, True, False]))


## 9. Optional: merge manual event selection

Use this once your manual-event table is ready. Two formats are supported:

1. **Hourly flags**: one row per timestamp, with columns like `timestamp`, `manual_event`, `manual_family`.
2. **Event intervals**: one row per event, with columns like `start`, `end`, `manual_family`, `event_id`.


In [ ]:
MANUAL_HOURLY_PATH = None
# Example:
# MANUAL_HOURLY_PATH = ROOT / "event_outputs" / "manual_event_hours.csv"

MANUAL_INTERVAL_PATH = None
# Example:
# MANUAL_INTERVAL_PATH = ROOT / "event_outputs" / "manual_event_intervals.csv"


def infer_time_column(df, candidates=("timestamp", "datetime", "date_time", "time", "start", "end")):
    ranked = []
    for c in df.columns:
        cl = c.lower()
        if any(token in cl for token in candidates):
            parsed = pd.to_datetime(df[c], errors="coerce")
            ranked.append((parsed.notna().sum(), c))
    if not ranked:
        return None
    return sorted(ranked, reverse=True)[0][1]


def merge_manual_hourly_flags(base_df, manual_path):
    manual = pd.read_csv(manual_path)
    manual.columns = [str(c).strip() for c in manual.columns]
    time_col_manual = infer_time_column(manual)
    if time_col_manual is None:
        raise ValueError("Could not infer timestamp column in manual hourly table.")
    manual["timestamp_local"] = localize_or_convert_time(manual[time_col_manual], tz=TZ)

    event_cols = [c for c in manual.columns if any(x in c.lower() for x in ["event", "flag", "manual"])]
    family_cols = [c for c in manual.columns if any(x in c.lower() for x in ["family", "class", "source", "label"])]

    keep = ["timestamp_local"]
    if event_cols: keep.append(event_cols[0])
    if family_cols: keep.append(family_cols[0])
    out = base_df.merge(manual[keep].dropna(subset=["timestamp_local"]), on="timestamp_local", how="left")

    out = out.rename(columns={event_cols[0]: "manual_event_flag"}) if event_cols else out.assign(manual_event_flag=False)
    out = out.rename(columns={family_cols[0]: "manual_family"}) if family_cols else out.assign(manual_family=np.nan)
    out["manual_event_flag"] = out["manual_event_flag"].fillna(False).astype(bool)
    out["manual_family"] = out["manual_family"].fillna("none")
    out["manual_event_id"] = "none"
    return out


def add_manual_interval_flags(base_df, interval_path):
    events = pd.read_csv(interval_path)
    events.columns = [str(c).strip() for c in events.columns]

    start_candidates = [c for c in events.columns if "start" in c.lower() or "begin" in c.lower()]
    end_candidates = [c for c in events.columns if "end" in c.lower() or "stop" in c.lower()]
    family_candidates = [c for c in events.columns if any(x in c.lower() for x in ["family", "class", "source", "label"])]
    id_candidates = [c for c in events.columns if "event" in c.lower() and "id" in c.lower()]

    if not start_candidates or not end_candidates:
        raise ValueError("Could not infer start/end columns in manual interval table.")

    start_col, end_col = start_candidates[0], end_candidates[0]
    family_col = family_candidates[0] if family_candidates else None
    id_col = id_candidates[0] if id_candidates else None

    events["start_local"] = localize_or_convert_time(events[start_col], tz=TZ)
    events["end_local"] = localize_or_convert_time(events[end_col], tz=TZ)

    out = base_df.copy()
    out["manual_event_flag"] = False
    out["manual_family"] = "none"
    out["manual_event_id"] = "none"

    for i, row in events.dropna(subset=["start_local", "end_local"]).iterrows():
        mask = (out["timestamp_local"] >= row["start_local"]) & (out["timestamp_local"] <= row["end_local"])
        fam = str(row[family_col]) if family_col else "manual_event"
        eid = str(row[id_col]) if id_col else f"manual_event_{i}"
        out.loc[mask, "manual_event_flag"] = True
        out.loc[mask & out["manual_family"].eq("none"), "manual_family"] = fam
        overlap = mask & out["manual_family"].ne("none") & out["manual_family"].ne(fam)
        out.loc[overlap, "manual_family"] = out.loc[overlap, "manual_family"] + ";" + fam
        out.loc[mask & out["manual_event_id"].eq("none"), "manual_event_id"] = eid
    return out


merged_manual = merged_flags.copy()
if MANUAL_HOURLY_PATH is not None:
    merged_manual = merge_manual_hourly_flags(merged_manual, MANUAL_HOURLY_PATH)
    print("Merged hourly manual flags:", MANUAL_HOURLY_PATH)
elif MANUAL_INTERVAL_PATH is not None:
    merged_manual = add_manual_interval_flags(merged_manual, MANUAL_INTERVAL_PATH)
    print("Merged interval manual flags:", MANUAL_INTERVAL_PATH)
else:
    print("No manual event file supplied yet. Skipping manual merge.")
    merged_manual["manual_event_flag"] = False
    merged_manual["manual_family"] = "none"
    merged_manual["manual_event_id"] = "none"

merged_manual.to_csv(OUT_DIR / "pmf_contributions_with_manual_flags.csv", index=False)
display(merged_manual[["timestamp_local", "manual_event_flag", "manual_family", *factor_cols]].head())


## 10. PMF vs manual event-selection overlap metrics


In [ ]:
def pmf_manual_overlap_metrics(df, factors, top_suffixes=("top05", "top01")):
    rows = []
    total_manual_hours = int(df["manual_event_flag"].sum())
    for f in factors:
        for suffix in top_suffixes:
            flag_col = f"{f}_{suffix}"
            high = df[flag_col].fillna(False)
            manual = df["manual_event_flag"].fillna(False)
            n_high = int(high.sum())
            n_overlap = int((high & manual).sum())
            rows.append({
                "factor": f,
                "factor_label": FACTOR_LABELS.get(f, f),
                "top_definition": suffix,
                "n_factor_high_hours": n_high,
                "n_manual_event_hours": total_manual_hours,
                "n_overlap_hours": n_overlap,
                "pct_high_hours_that_are_manual": 100 * n_overlap / n_high if n_high else np.nan,
                "pct_manual_hours_captured_by_factor_high": 100 * n_overlap / total_manual_hours if total_manual_hours else np.nan,
            })
    return pd.DataFrame(rows)

manual_overlap = pmf_manual_overlap_metrics(merged_manual, factor_cols)
manual_overlap.to_csv(OUT_DIR / "pmf_manual_overlap_metrics.csv", index=False)
display(manual_overlap)

family_rows = []
for f in factor_cols:
    for suffix in ["top05", "top01"]:
        flag_col = f"{f}_{suffix}"
        sub = merged_manual[merged_manual[flag_col]].copy()
        if sub.empty:
            continue
        for fam, n in sub["manual_family"].value_counts(dropna=False).items():
            family_rows.append({
                "factor": f,
                "factor_label": FACTOR_LABELS.get(f, f),
                "top_definition": suffix,
                "manual_family": fam,
                "n_high_hours": int(n),
                "pct_of_factor_high_hours": 100 * n / len(sub),
            })

family_overlap = pd.DataFrame(family_rows)
family_overlap.to_csv(OUT_DIR / "pmf_manual_family_overlap.csv", index=False)
display(family_overlap.sort_values(["factor", "top_definition", "n_high_hours"], ascending=[True, True, False]).head(50))


## 11. Export top PMF hours for manual inspection


In [ ]:
top_rows = []
for f in factor_cols:
    for suffix in ["top05", "top01"]:
        flag_col = f"{f}_{suffix}"
        cols = [
            "timestamp_local", "sample_num", "SampleID_pmf", "daynight", "hour", "weekday_weekend", "season", "month_name",
            "known_event_window", "manual_event_flag", "manual_family", f,
        ]
        sub = merged_manual.loc[merged_manual[flag_col], cols].copy()
        sub["factor"] = f
        sub["factor_label"] = FACTOR_LABELS.get(f, f)
        sub["top_definition"] = suffix
        sub = sub.rename(columns={f: "factor_contribution"})
        top_rows.append(sub)

top_hours = pd.concat(top_rows, ignore_index=True)
top_hours.to_csv(OUT_DIR / "pmf_top_factor_hours_for_manual_inspection.csv", index=False)
display(top_hours.sort_values(["factor", "top_definition", "factor_contribution"], ascending=[True, True, False]).head(50))


## Interpretation checklist

Use this notebook to answer:

- **Fe–Mn-rich factor:** do top hours overlap manual Fe–Mn / Fe–Zn / industrial metal events? Are they night-enhanced or multi-hour plumes?
- **Zn-rich factor:** do top hours overlap manual Zn–As–Se–Pb or Zn-rich events? Are they recurrent outside fireworks/wildfire windows?
- **Ca–Fe–Ti factor:** does it show daytime/weekday/seasonal behavior consistent with dust/road/urban activity?
- **S-rich factor:** does it behave like broad regional/background aerosol, rather than isolated local spikes?
- **K-rich and Cu–Ba/Sr-rich factors:** do they spike during fireworks/wildfire windows? Interpret these qualitatively unless manual/event evidence is strong.
